# NBA Text-to-SQL

A pipeline that converts natural language questions into PostgreSQL queries for an NBA statistics database.

This notebook showcases:
1. How data is collected from the NBA API
2. Inference with our fine-tuned T5 baseline model
3. Examples where the model succeeds and fails
4. Evaluation results

**Model:** https://huggingface.co/EthanGC/nba-t5-text-to-sql

```bash
pip install transformers torch sentencepiece nba_api
```

## 1. Data Collection via NBA API

All data in the database was collected using the `nba_api` package, which pulls statistics directly from the NBA's official data endpoints. Below is an example fetching season totals for the 2023-24 season — the same call used to populate the `player_general_traditional_total` table.

In [1]:
import time
from nba_api.stats.endpoints import LeagueDashPlayerStats

stats = LeagueDashPlayerStats(
    season="2023-24",
)

df_api = stats.get_data_frames()[0]
cols = ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "PTS", "REB", "AST", "STL", "BLK", "TOV", "FG3A", "FG3_PCT"]
print(f"Players in 2023-24: {len(df_api)}")
df_api[cols].sort_values("PTS", ascending=False).head(10)

Players in 2023-24: 572


,PLAYER_NAME,TEAM_ABBREVIATION,GP,PTS,REB,AST,STL,BLK,TOV,FG3A,FG3_PCT
372,Luka Dončić,DAL,70,2370,647,686,99,38,282,744,0.382
501,Shai Gilgeous-Alexander,OKC,75,2254,415,465,150,67,162,269,0.353
184,Giannis Antetokounmpo,MIL,73,2222,841,476,87,79,250,124,0.274
233,Jalen Brunson,NYK,77,2212,278,519,70,13,186,526,0.401
434,Nikola Jokić,DEN,79,2085,976,708,108,68,237,231,0.359
29,Anthony Edwards,MIN,79,2049,430,405,101,42,241,532,0.357
337,Kevin Durant,PHX,75,2032,495,378,69,91,244,407,0.413
268,Jayson Tatum,BOS,74,1987,601,364,75,43,188,609,0.376
125,De'Aaron Fox,SAC,74,1966,340,418,150,31,194,580,0.369
509,Stephen Curry,GSW,74,1956,330,379,54,28,210,876,0.408


This raw data is cleaned and inserted into PostgreSQL across six tables: `player`, `team`, `game`, `player_game_log`, `player_season`, and `player_general_traditional_total`. The text-to-SQL model is trained to query this exact schema.

## 2. Load the Baseline Model

In [2]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

HF_MODEL = "EthanGC/nba-t5-text-to-sql"

tokenizer = T5Tokenizer.from_pretrained(HF_MODEL)
model = T5ForConditionalGeneration.from_pretrained(HF_MODEL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Loaded on {device}")

c:\Users\Ethan\homework\cs175\nba-text-to-sql\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded on cuda


In [3]:
def predict(question: str) -> str:
    prefixed = f"translate English to SQL: {question}"
    inputs = tokenizer(prefixed, return_tensors="pt",
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=256,
                                 num_beams=5, repetition_penalty=1.5)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def edit_distance(a: str, b: str) -> int:
    """Character-level Levenshtein distance between two strings."""
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            temp = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = temp
    return dp[m]

## 3. Where the model works well

In [4]:
good_examples = [
    {
        "question": "top 5 scorers in 2023",
        "gold":     "SELECT p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id WHERE ps.season_id = 22022 ORDER BY ps.pts DESC LIMIT 5;"
    },
    {
        "question": "how many players shot over 40 percent from three in 2018 with at least 100 attempts",
        "gold":     "SELECT COUNT(*) AS num_players FROM player_general_traditional_total WHERE season_id = 22017 AND fg3_pct > 0.40 AND fg3a >= 100;"
    },
    {
        "question": "players from Duke",
        "gold":     "SELECT player_name FROM player WHERE college = 'Duke' ORDER BY player_name;"
    }
]

for ex in good_examples:
    pred = predict(ex["question"])
    dist = edit_distance(ex["gold"], pred)
    print(f"Q:         {ex['question']}")
    print(f"Gold:      {ex['gold']}")
    print(f"Pred:      {pred}")
    print(f"Edit dist: {dist}")
    print()

Q:         top 5 scorers in 2023
Gold:      SELECT p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id WHERE ps.season_id = 22022 ORDER BY ps.pts DESC LIMIT 5;
Pred:      SELECT p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id WHERE ps.season_id = 22022 ORDER BY ps.pts DESC LIMIT 5;
Edit dist: 0

Q:         how many players shot over 40 percent from three in 2018 with at least 100 attempts
Gold:      SELECT COUNT(*) AS num_players FROM player_general_traditional_total WHERE season_id = 22017 AND fg3_pct > 0.40 AND fg3a >= 100;
Pred:      SELECT COUNT(*) AS num_players FROM player_general_traditional_total WHERE season_id = 22017 AND fg3_pct > 0.40 AND fg3a >= 100;
Edit dist: 0

Q:         players from Duke
Gold:      SELECT player_name FROM player WHERE college = 'Duke' ORDER BY player_name;
Pred:      SELECT player_name FROM player WHERE college = 'Duke' ORDER BY player_name;
Edit dist: 0



## 4. Where the model struggles

In [5]:
hard_examples = [
    {
        "question": "who was the best scorer for the cavaliers in each season",
        "gold":     "SELECT pg.season_id, p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id JOIN team t ON ps.team_id = t.team_id JOIN player_general_traditional_total pg ON p.player_id = pg.player_id AND ps.season_id = pg.season_id WHERE t.nickname = 'Cavaliers' AND ps.pts = (SELECT MAX(ps2.pts) FROM player_season ps2 WHERE ps2.team_id = ps.team_id AND ps2.season_id = ps.season_id) ORDER BY pg.season_id;"
    },
    {
        "question": "LeBron James points per game in 2020",
        "gold":     "SELECT ps.pts FROM player_season ps JOIN player p ON ps.player_id = p.player_id WHERE p.player_name = 'LeBron James' AND ps.season_id = 22019;"
    }
]

for ex in hard_examples:
    pred = predict(ex["question"])
    dist = edit_distance(ex["gold"], pred)
    print(f"Q:         {ex['question']}")
    print(f"Gold:      {ex['gold']}")
    print(f"Pred:      {pred}")
    print(f"Edit dist: {dist}")
    print()

Q:         who was the best scorer for the cavaliers in each season
Gold:      SELECT pg.season_id, p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id JOIN team t ON ps.team_id = t.team_id JOIN player_general_traditional_total pg ON p.player_id = pg.player_id AND ps.season_id = pg.season_id WHERE t.nickname = 'Cavaliers' AND ps.pts = (SELECT MAX(ps2.pts) FROM player_season ps2 WHERE ps2.team_id = ps.team_id AND ps2.season_id = ps.season_id) ORDER BY pg.season_id;
Pred:      SELECT p.player_name, ps.pts FROM player p JOIN player_season ps ON p.player_id = ps.player_id JOIN team t ON ps.team_id = t.team_id WHERE t.nickname = 'Cavaliers' ORDER BY ps.pts DESC LIMIT 1;
Edit dist: 245

Q:         LeBron James points per game in 2020
Gold:      SELECT ps.pts FROM player_season ps JOIN player p ON ps.player_id = p.player_id WHERE p.player_name = 'LeBron James' AND ps.season_id = 22019;
Pred:      SELECT p.player_name, ps.pts FROM player p JOIN player_season